# NB01: Data Extraction

Extract all PGP gene data, environment labels, taxonomy, pangenome stats, and GapMind tryptophan scores. Foundation for all downstream notebooks.

**Steps:**
1. Extract PGP gene clusters × gene_cluster (core/aux/singleton + species)
2. Build species-level PGP trait matrix (wide format)
3. Classify per-genome environment from ncbi_isolation_source
4. Build species-level majority-vote environment labels
5. Extract pangenome openness stats
6. Extract GapMind tryptophan completeness
7. Validation checkpoint

**Requires**: Spark (on BERDL JupyterHub)

**Outputs**: `data/pgp_clusters.csv`, `data/species_pgp_matrix.csv`, `data/genome_environment.csv`, `data/species_environment.csv`, `data/pangenome_stats.csv`, `data/trp_completeness.csv`

In [ ]:
import os
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore', category=FutureWarning)

_here = os.path.abspath('')
if os.path.basename(_here) == 'notebooks':
    REPO = os.path.abspath(os.path.join(_here, '..', '..', '..'))
elif os.path.exists(os.path.join(_here, 'projects', 'pgp_pangenome_ecology')):
    REPO = _here
else:
    REPO = os.path.abspath(os.path.join(_here, '..', '..', '..'))

PROJECT = os.path.join(REPO, 'projects', 'pgp_pangenome_ecology')
DATA = os.path.join(PROJECT, 'data')
FIGURES = os.path.join(PROJECT, 'figures')

os.makedirs(DATA, exist_ok=True)
os.makedirs(FIGURES, exist_ok=True)

print(f'REPO: {REPO}')
print(f'DATA: {DATA}')

## 1. Start Spark and extract PGP gene clusters

In [ ]:
spark = get_spark_session()

# Filter bakta_annotations by gene name (132M rows — filter first, then join)
pgp_clusters = spark.sql("""
    SELECT ba.gene_cluster_id, ba.gene, ba.product, ba.kegg_orthology_id,
           gc.gtdb_species_clade_id, gc.is_core, gc.is_auxiliary, gc.is_singleton
    FROM kbase_ke_pangenome.bakta_annotations ba
    JOIN kbase_ke_pangenome.gene_cluster gc ON ba.gene_cluster_id = gc.gene_cluster_id
    WHERE ba.gene IN ('nifH','nifD','nifK','acdS','pqqA','pqqB','pqqC','pqqD','pqqE',
                      'ipdC','hcnA','hcnB','hcnC')
    AND ba.gene IS NOT NULL
""").toPandas()

print(f'PGP gene clusters: {len(pgp_clusters):,}')
print(f'Unique species: {pgp_clusters["gtdb_species_clade_id"].nunique():,}')
print(f'\nPer-gene cluster counts:')
print(pgp_clusters['gene'].value_counts().to_string())

In [ ]:
# Core/aux/singleton breakdown
print('Core/auxiliary/singleton breakdown:')
print(f'  is_core:      {pgp_clusters["is_core"].sum():,}')
print(f'  is_auxiliary: {pgp_clusters["is_auxiliary"].sum():,}')
print(f'  is_singleton: {pgp_clusters["is_singleton"].sum():,}')

# Validation: nifH clusters should be ~1,913
nifh_count = (pgp_clusters['gene'] == 'nifH').sum()
print(f'\nValidation: nifH clusters = {nifh_count:,} (expected ~1,913)')

# Save raw cluster data
pgp_clusters.to_csv(os.path.join(DATA, 'pgp_clusters.csv'), index=False)
print(f'Saved pgp_clusters.csv')

## 2. Build species-level PGP trait matrix

In [ ]:
# Species × gene presence/count (long format first)
species_gene = pgp_clusters.groupby(['gtdb_species_clade_id', 'gene']).agg(
    n_clusters=('gene_cluster_id', 'nunique'),
    n_core=('is_core', 'sum'),
    n_aux=('is_auxiliary', 'sum'),
    n_singleton=('is_singleton', 'sum'),
).reset_index()

# Pivot to wide format: species × gene → cluster count
PGP_GENES = ['nifH', 'nifD', 'nifK', 'acdS', 'pqqA', 'pqqB', 'pqqC', 'pqqD', 'pqqE',
             'ipdC', 'hcnA', 'hcnB', 'hcnC']

species_pgp = species_gene.pivot_table(
    index='gtdb_species_clade_id',
    columns='gene',
    values='n_clusters',
    fill_value=0
).reset_index()

# Ensure all PGP genes are present as columns
for g in PGP_GENES:
    if g not in species_pgp.columns:
        species_pgp[g] = 0

# Add binary presence columns
for g in PGP_GENES:
    species_pgp[f'{g}_present'] = (species_pgp[g] > 0).astype(int)

# Total PGP gene count per species
species_pgp['n_pgp_genes'] = species_pgp[[f'{g}_present' for g in PGP_GENES]].sum(axis=1)

print(f'Species × PGP gene matrix: {species_pgp.shape}')
print(f'Species with any PGP gene: {len(species_pgp):,}')
print(f'\nSpecies with each gene:')
for g in PGP_GENES:
    n = species_pgp[f'{g}_present'].sum()
    print(f'  {g:8s}: {n:6,}')

species_pgp.to_csv(os.path.join(DATA, 'species_pgp_matrix.csv'), index=False)
print(f'\nSaved species_pgp_matrix.csv')

## 3. Per-genome environment classification

In [ ]:
# Extract genome environment from gtdb_metadata (direct column, not EAV)
print('Extracting per-genome environment from gtdb_metadata...')

env_df = spark.sql("""
    SELECT m.accession AS genome_id, m.ncbi_isolation_source,
           t.phylum, t.class, t.order, t.family, t.genus, t.species,
           g.gtdb_species_clade_id
    FROM kbase_ke_pangenome.gtdb_metadata m
    JOIN kbase_ke_pangenome.genome g ON m.accession = g.genome_id
    JOIN kbase_ke_pangenome.gtdb_taxonomy_r214v1 t ON g.genome_id = t.genome_id
    WHERE m.ncbi_isolation_source IS NOT NULL
""").toPandas()

print(f'Genomes with ncbi_isolation_source: {len(env_df):,}')
print(f'Sample isolation sources:')
print(env_df['ncbi_isolation_source'].value_counts().head(10).to_string())

In [ ]:
# Regex-based environment classification
SOIL_PATTERNS = re.compile(
    r'soil|rhizospher|root.nodule|nodule|rhizobium|\bplant\b|\broot\b|rhizo',
    re.IGNORECASE
)
HOST_PATTERNS = re.compile(
    r'human|clinical|\bblood\b|patient|hospital|stool|feces|fecal|gut|intestin',
    re.IGNORECASE
)
AQUATIC_PATTERNS = re.compile(
    r'ocean|marine|\bsea\b|\blake\b|\briver\b|aquatic|\bwater\b|freshwater|estuar',
    re.IGNORECASE
)


def classify_env(iso_source):
    if pd.isna(iso_source) or str(iso_source).strip() == '':
        return 'unknown'
    s = str(iso_source)
    if SOIL_PATTERNS.search(s):
        return 'soil_rhizosphere'
    if HOST_PATTERNS.search(s):
        return 'host_clinical'
    if AQUATIC_PATTERNS.search(s):
        return 'marine_aquatic'
    return 'other'


env_df['env_class'] = env_df['ncbi_isolation_source'].apply(classify_env)

print('Per-genome environment classification:')
print(env_df['env_class'].value_counts().to_string())
classified_frac = (env_df['env_class'] != 'unknown').mean()
print(f'\nClassified: {classified_frac*100:.1f}% of genomes with isolation_source')

## 4. Species-level majority-vote environment

In [ ]:
# Species-level dominant environment
classified = env_df[env_df['env_class'] != 'unknown'].copy()

# Votes per species × env_class
votes = classified.groupby(['gtdb_species_clade_id', 'env_class']).size().reset_index(name='count')
totals = classified.groupby('gtdb_species_clade_id')['env_class'].count().reset_index(name='n_classified')

# Dominant environment per species
idx = votes.groupby('gtdb_species_clade_id')['count'].idxmax()
species_env = votes.loc[idx][['gtdb_species_clade_id', 'env_class', 'count']].copy()
species_env = species_env.merge(totals, on='gtdb_species_clade_id')
species_env['majority_frac'] = species_env['count'] / species_env['n_classified']
species_env.columns = ['gtdb_species_clade_id', 'dominant_env', 'dominant_count',
                        'n_classified_genomes', 'majority_frac']

# Also store genome counts per env per species (for H2 sensitivity analysis)
species_env_counts = votes.pivot_table(
    index='gtdb_species_clade_id', columns='env_class', values='count', fill_value=0
).reset_index()
species_env = species_env.merge(species_env_counts, on='gtdb_species_clade_id', how='left')

print(f'Species with environment labels: {len(species_env):,}')
print(f'\nSpecies-level environment distribution:')
print(species_env['dominant_env'].value_counts().to_string())
print(f'\nMajority fraction: mean={species_env["majority_frac"].mean():.2f}, '
      f'median={species_env["majority_frac"].median():.2f}')

env_df.to_csv(os.path.join(DATA, 'genome_environment.csv'), index=False)
species_env.to_csv(os.path.join(DATA, 'species_environment.csv'), index=False)
print(f'\nSaved genome_environment.csv and species_environment.csv')

## 5. Pangenome openness stats

In [ ]:
pangenome_stats = spark.sql("""
    SELECT gtdb_species_clade_id,
           CAST(no_genomes AS INT) AS no_genomes,
           CAST(no_core AS INT) AS no_core,
           CAST(no_aux_genome AS INT) AS no_aux_genome,
           CAST(no_singleton_gene_clusters AS INT) AS no_singleton_gene_clusters,
           CAST(no_gene_clusters AS INT) AS no_gene_clusters
    FROM kbase_ke_pangenome.pangenome
    WHERE no_gene_clusters > 0
""").toPandas()

pangenome_stats['singleton_fraction'] = (
    pangenome_stats['no_singleton_gene_clusters'] / pangenome_stats['no_gene_clusters']
)
pangenome_stats['accessory_fraction'] = (
    pangenome_stats['no_aux_genome'] / pangenome_stats['no_gene_clusters']
)
pangenome_stats['core_fraction'] = (
    pangenome_stats['no_core'] / pangenome_stats['no_gene_clusters']
)

print(f'Species with pangenome stats: {len(pangenome_stats):,}')
print(f'Genome-wide baseline:')
print(f'  Mean core fraction:      {pangenome_stats["core_fraction"].mean():.3f}')
print(f'  Mean accessory fraction: {pangenome_stats["accessory_fraction"].mean():.3f}')
print(f'  Mean singleton fraction: {pangenome_stats["singleton_fraction"].mean():.3f}')

pangenome_stats.to_csv(os.path.join(DATA, 'pangenome_stats.csv'), index=False)
print(f'Saved pangenome_stats.csv')

## 6. GapMind tryptophan completeness

In [ ]:
# GapMind trp completeness for H4
# Also fetch tyr as a negative control
trp_completeness = spark.sql("""
    SELECT clade_name AS gtdb_species_clade_id, score_simplified AS trp_complete
    FROM kbase_ke_pangenome.gapmind_pathways
    WHERE pathway = 'trp' AND metabolic_category = 'aa'
    AND sequence_scope = 'core'
""").toPandas()

print(f'Species with trp GapMind score: {len(trp_completeness):,}')
print(f'trp_complete distribution:')
print(trp_completeness['trp_complete'].value_counts().to_string())

# Tyrosine as negative control for NB05
try:
    tyr_completeness = spark.sql("""
        SELECT clade_name AS gtdb_species_clade_id, score_simplified AS tyr_complete
        FROM kbase_ke_pangenome.gapmind_pathways
        WHERE pathway = 'tyr' AND metabolic_category = 'aa'
        AND sequence_scope = 'core'
    """).toPandas()
    trp_completeness = trp_completeness.merge(tyr_completeness, on='gtdb_species_clade_id', how='outer')
    print(f'\nAlso loaded tyr as negative control: {tyr_completeness["tyr_complete"].value_counts().to_string()}')
except Exception as e:
    print(f'tyr pathway not available: {e}')

trp_completeness.to_csv(os.path.join(DATA, 'trp_completeness.csv'), index=False)
print(f'Saved trp_completeness.csv')

## 7. Validation checkpoint

In [ ]:
print('=== DATA VALIDATION CHECKPOINT ===')
print()

# (a) PGP cluster count validation
nifh = (pgp_clusters['gene'] == 'nifH').sum()
print(f'(a) nifH clusters: {nifh:,} (expected ~1,913)')
print(f'    Total PGP clusters: {len(pgp_clusters):,}')

# (b) Isolation source coverage
total_genomes_approx = 293059  # from taxonomy table
iso_coverage = len(env_df) / total_genomes_approx * 100
print(f'\n(b) ncbi_isolation_source coverage: {len(env_df):,} / ~293K genomes ({iso_coverage:.1f}%)')
print(f'    Classifiable (non-unknown): {classified_frac*100:.1f}% of those')

# (c) Species-level environment coverage
print(f'\n(c) Species with dominant env: {len(species_env):,}')
print(f'    soil_rhizosphere: {(species_env["dominant_env"] == "soil_rhizosphere").sum():,}')

# (d) Pangenome stats coverage
print(f'\n(d) Species with pangenome stats: {len(pangenome_stats):,}')

# (e) GapMind coverage
print(f'\n(e) Species with trp GapMind: {len(trp_completeness):,}')

# (f) Known PGPB genera — sanity check
known_pgpb = ['Rhizobium', 'Azospirillum', 'Pseudomonas', 'Bacillus', 'Mesorhizobium']
print(f'\n(f) Known PGPB genera PGP gene counts:')
if 'genus' in env_df.columns:
    sp_with_tax = species_pgp.merge(
        env_df.drop_duplicates('gtdb_species_clade_id')[['gtdb_species_clade_id', 'genus']],
        on='gtdb_species_clade_id', how='left'
    )
    for genus in known_pgpb:
        sub = sp_with_tax[sp_with_tax['genus'].str.contains(genus, na=False)]
        if len(sub) > 0:
            med_pgp = sub['n_pgp_genes'].median()
            print(f'  {genus:20s}: {len(sub):4d} species, median PGP genes = {med_pgp:.1f}')

In [ ]:
# Summary figure: PGP gene prevalence across species
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: per-gene species count
ax = axes[0]
gene_counts = {g: species_pgp[f'{g}_present'].sum() for g in PGP_GENES}
gene_df = pd.Series(gene_counts).sort_values(ascending=True)
colors = ['#2196F3' if 'nif' in g else '#4CAF50' if 'pqq' in g
          else '#FF9800' if g in ('ipdC', 'acdS') else '#9C27B0'
          for g in gene_df.index]
ax.barh(gene_df.index, gene_df.values, color=colors)
ax.set_xlabel('Number of species')
ax.set_title('PGP gene prevalence across species')
ax.set_xlim(0, gene_df.max() * 1.15)
for i, (g, n) in enumerate(gene_df.items()):
    ax.text(n + gene_df.max() * 0.01, i, f'{n:,}', va='center', fontsize=8)

# Right: number of PGP genes per species histogram
ax = axes[1]
ax.hist(species_pgp['n_pgp_genes'], bins=range(0, species_pgp['n_pgp_genes'].max() + 2),
        color='steelblue', edgecolor='white', alpha=0.8)
ax.set_xlabel('Number of PGP genes per species')
ax.set_ylabel('Species count')
ax.set_title('PGP gene richness per species')
ax.set_yscale('log')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES, 'nb01_pgp_prevalence.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/nb01_pgp_prevalence.png')

In [ ]:
print('=== NB01 Summary ===')
print(f'PGP clusters extracted: {len(pgp_clusters):,} across {species_pgp["gtdb_species_clade_id"].nunique():,} species')
print(f'Genome environment classified: {classified_frac*100:.1f}% of genomes with isolation_source')
print(f'Species with env label: {len(species_env):,}')
print(f'Pangenome stats: {len(pangenome_stats):,} species')
print(f'GapMind trp: {len(trp_completeness):,} species')
print(f'\nReady for NB02 (co-occurrence), NB03 (env enrichment), NB04 (core/accessory), NB05 (trp-IAA)')